# Milestone 1 - Basic AI Model Training


## Environment Setup & Hardware Verification

Hardware detection and package installation for running locally or in Colab.

In [ ]:
# System and hardware verification - putting this in due to the numerous issues I have had on my local machine with this.
import sys
import platform
import os

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")

try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed; skipping GPU detection.")

In [ ]:
%pip install -q --upgrade pip
%pip install -q pandas numpy scipy matplotlib seaborn scikit-learn pyarrow fastparquet openpyxl google-cloud-storage

In [ ]:
# Core dependencies verification
try:
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sklearn
    import pyarrow
    import openpyxl
    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"Libraries not yet installed in current runtime ({e}).")
    print("Run the %pip install cell above when running in Google Colab.")

In [ ]:
from pathlib import Path

# Project root directory setup
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

for p in [DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"Directory ready: {p.relative_to(BASE_DIR)}")

# 1. Raw Data Storage

The raw data for this project is `attack_dataset.csv`. This csv file (in this repo) will be copied into the notebook and then into the bucket.

### How I came up with this approach:

- **ephemeral nature of colab runtimes**: Every time you exit your Colab session (or let it go idle), the runtime gets destroyed. The same applies to its local disk space. If all your raw files were stored solely on the Colab instance's disk, they'd get lost after each run. By storing the base file in a GCP bucket, I'm making sure it persists through multiple runs of the notebook.
- **immutability of the raw data**: Per the lectures, raw data should never change. A raw data file could be useful in case a problem comes to light later when cleaning/preprocessing features in a subsequent step; if the raw file has been preserved from the beginning, I'll always have something I can use to rerun everything from a clean slate without worrying about corrupting the data. In my code, I set the local cached version to be read only so I can't accidentally changes it.
- **keeping data out of git** : It is generally considered bad practice to include raw data directly in the Git repository as was shown in lecture 2. The Git repository is meant to track versions of code/configurations/checksums, whereas the raw data goes into the storage system.
- **avoid I/O overhead** : With Lecture 2 in mind, reading lines from the network every time I train an epoch/model or preprocess some subset of a larger dataset can lead to huge I/O overhead and leaves my compute resources unused. By caching `attack_dataset.csv` locally to the NVMe SSD (once) at `/data/raw`, we enable fast sequential access for both Pandas and PyTorch.

In [ ]:
from pathlib import Path
import os
import stat

# Storage configuration
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_FILE = RAW_DATA_DIR / "attack_dataset.csv"

GCS_PROJECT_ID = os.environ.get("GCP_PROJECT", "setu-ai-hdip")
GCS_BUCKET_NAME = "setu-hdip-ai-raw"
GCS_REGION = "europe-west1"
GCS_BLOB_PATH = "cybersecurity/attack_dataset.csv"
REMOTE_GCS_URI = f"gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}"

print(f"Target GCS Bucket: gs://{GCS_BUCKET_NAME}")
print(f"Local Raw Path:    {RAW_DATA_FILE}")

# Spin up GCS databucket and sync raw file
# (uncomment auth in Colab if authenticating for the first time)
# from google.colab import auth
# auth.authenticate_user()

try:
    from google.cloud import storage
    storage_client = storage.Client(project=GCS_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    
    if not bucket.exists():
        print(f"Spinning up GCS bucket: gs://{GCS_BUCKET_NAME} ({GCS_REGION})...")
        bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=GCS_REGION)
        print(f"Bucket ready: gs://{GCS_BUCKET_NAME}")
    else:
        print(f"Connected to existing bucket: gs://{GCS_BUCKET_NAME}")
        
    # Sync to GCS if local file is present
    if RAW_DATA_FILE.exists():
        blob = bucket.blob(GCS_BLOB_PATH)
        if not blob.exists():
            print(f"Uploading {RAW_DATA_FILE.name} to gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}...")
            blob.upload_from_filename(str(RAW_DATA_FILE))
            print("Upload to bucket complete.")
        else:
            print(f"Blob already exists in bucket: gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}")
except ImportError:
    print("Note: google-cloud-storage not installed in current environment.")
except Exception as e:
    print(f"GCS bucket status note: {e}")

# Enforce read-only immutability on local raw file
if RAW_DATA_FILE.exists():
    try:
        os.chmod(RAW_DATA_FILE, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
        print("Permissions: Set to read-only (immutable).")
    except Exception as e:
        print(f"Note on permissions: {e}")
else:
    print(f"Local raw file '{RAW_DATA_FILE.name}' not found yet. It can be uploaded or synced in Section 5 (Data Access).")

## 2. Processed Data Storage & File Formats

Where processed datasets will live and the file formats chosen (e.g. Parquet vs CSV).

In [ ]:
from pathlib import Path

# Processed data directory
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Processed data directory ready: {PROCESSED_DATA_DIR.relative_to(BASE_DIR)}")

## 3. Database / Object Storage Decision

Evaluation of database vs object storage vs filesystem for this project's requirements.

## 4. Data Versioning



In [ ]:
from pathlib import Path
import hashlib
import json
import time

DATASET_VERSION = "v1.0"

# Compute SHA-256 checksum and generate version manifest
if RAW_DATA_FILE.exists():
    hasher = hashlib.sha256()
    with open(RAW_DATA_FILE, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    sha256_hash = hasher.hexdigest()
    file_size_kb = RAW_DATA_FILE.stat().st_size / 1024
    
    version_metadata = {
        "dataset_name": "attack_dataset.csv",
        "version": DATASET_VERSION,
        "sha256": sha256_hash,
        "size_kb": round(file_size_kb, 2),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "storage_uri": REMOTE_GCS_URI
    }
    
    manifest_file = RAW_DATA_DIR / "dataset_manifest.json"
    with open(manifest_file, "w") as f:
        json.dump(version_metadata, f, indent=2)
        
    print(f"Dataset Version: {DATASET_VERSION}")
    print(f"SHA-256 Hash:    {sha256_hash}")
    print(f"File Size:       {file_size_kb:.2f} KB")
    print(f"Version Manifest saved to: {manifest_file.relative_to(BASE_DIR)}")
else:
    print(f"Note: '{RAW_DATA_FILE.name}' not found yet. Run Section 5 (Data Access) to upload or sync the dataset, then re-run this cell.")

## 5. Data Access



In [ ]:
from pathlib import Path
import os
import shutil

# 1. Google Drive mount (uncomment in Colab if using Drive persistence)
# from google.colab import drive
# drive.mount('/content/drive')

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/HDip_AI_Project")
GDRIVE_RAW_FILE = GDRIVE_PROJECT_DIR / "data/raw/attack_dataset.csv"

# 2. Check Drive mount first; sync to local cache if present
if GDRIVE_RAW_FILE.exists() and not RAW_DATA_FILE.exists():
    shutil.copy(GDRIVE_RAW_FILE, RAW_DATA_FILE)
    print(f"Synced raw data from Drive to local cache: {RAW_DATA_FILE}")

# 3. If still missing, prompt user for CSV file upload
if not RAW_DATA_FILE.exists():
    try:
        from google.colab import files
        print("Please select and upload your 'attack_dataset.csv' file:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = RAW_DATA_DIR / fname
            with open(dest, "wb") as f:
                f.write(data)
            print(f"Uploaded and saved: {dest}")
            if fname != RAW_DATA_FILE.name and not RAW_DATA_FILE.exists():
                shutil.copy(dest, RAW_DATA_FILE)
    except ImportError:
        # Fallback for local run outside Colab
        print(f"Running outside Colab. '{RAW_DATA_FILE.name}' not found in {RAW_DATA_DIR}.")
        print("Place the CSV file into './data/raw/' to proceed.")
else:
    print(f"Raw data available for access at: {RAW_DATA_FILE}")

## 6. Data Split / Validation Strategy

Train/dev/test partitioning or cross-validation strategy, avoiding data leakage.

## 7. Feature Description

Key features used from `attack_dataset.csv` and what each feature represents.

## 8. Data Types and Formats

Breakdown of data types (numerical, categorical, timestamps) and schema handling.

## 9. Reproducibility of Data Collection

Data sources, collection procedures, and parameters needed to reproduce the dataset.

## 10. Reproducibility of Preprocessing

Detailed steps to clean, filter, and transform the raw data into processed features.

## Verification & Baseline Pipeline Test

This should be checks to make sure everything is spun up properly.